In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/non_exploratory/check'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/matching_graph_paths.json')


# Circuit and Matching Graph

In [ ]:
from pathlib import Path
import sys
# Legacy class exports retain the coordinate methods used by these diagrams.
from codes.ca_code import CA_5_2_3, CA_9_2_5

%load_ext autoreload
%autoreload 2


In [ ]:
import numpy as np
import stim

def build_circuit(code, error_rate: float=0.001, rounds: int=3) -> stim.Circuit:
    h = code.H
    m, n = h.shape
    qubit_coords = code.qubit_coords()
    stab_coords = code.stabilizer_coords()
    data_qubits = list(range(n))
    ancilla_qubits = list(range(n, n + m))
    circuit = stim.Circuit()

    for i in range(n):
        circuit.append('QUBIT_COORDS', [i], qubit_coords[i])
    circuit.append('R', data_qubits + ancilla_qubits)
    circuit.append('TICK')

    for r in range(rounds):
        circuit.append('X_ERROR', data_qubits, error_rate)
        cnot = np.where(h == 1)
        for i, j in zip(*cnot):
            circuit.append('CNOT', [j, n + i])
            circuit.append('DEPOLARIZE2', [j, n + i], error_rate)

        circuit.append('TICK')
        circuit.append('M', ancilla_qubits)
        for i in range(m):
            coords = [*stab_coords[i], 3 * r]
            if r == 0:
                circuit.append('DETECTOR', [stim.target_rec(-m + i)], coords)
            else:
                circuit.append('DETECTOR', [stim.target_rec(-m + i), stim.target_rec(-2 * m + i)], coords)

        if r < rounds - 1:
            circuit.append('R', ancilla_qubits)
            circuit.append('TICK')

    circuit.append('M', data_qubits)
    circuit.append('TICK')
    for i in range(m):
        coords = [*stab_coords[i], 3 * rounds]
        targets = [stim.target_rec(-n - m + i)]
        for j in range(n):
            if h[i, j] == 1:
                targets.append(stim.target_rec(-n + j))

        circuit.append('DETECTOR', targets, coords)

    obs = np.where(code.L_Z == 1)
    for i, j in zip(*obs):
        circuit.append('OBSERVABLE_INCLUDE', [stim.target_rec(-n + j)], i)

    return circuit


In [ ]:
code1 = CA_5_2_3()
circuit = build_circuit(code1, rounds=3)
circuit.detector_error_model()


In [ ]:
code1 = CA_5_2_3()
circuit = build_circuit(code1, rounds=1)
with open(str(_data_paths.output('circ_code1_svg')), 'w') as f:
    f.write(str(circuit.diagram('timeline-svg')))
diagram = circuit.detector_error_model().diagram('matchgraph-svg')

with open(str(_data_paths.output('mg_code1_svg')), 'w') as f:
    f.write(str(diagram))
diagram


In [ ]:
code2 = CA_9_2_5()
circuit = build_circuit(code2, rounds=3)
with open(str(_data_paths.output('circ_code2_svg')), 'w') as f:
    f.write(str(circuit.diagram('timeline-svg')))
diagram = circuit.detector_error_model().diagram('matchgraph-svg')

with open(str(_data_paths.output('mg_code2_svg')), 'w') as f:
    f.write(str(diagram))
diagram


## Test

In [ ]:
import sys
from pathlib import Path
from src.noises_ca import CircuitBuilder, ErrorRate
# Legacy class exports retain the coordinate methods used by these diagrams.
from codes.ca_code import CA_9_2_5, CA_5_2_3

%load_ext autoreload
%autoreload 2


In [ ]:
code = CA_9_2_5()
builder = CircuitBuilder(code)
circuit = builder.get_circuit(noise_model='phenomenological', error_rate=ErrorRate(p=0.01), d_rounds=2)
dem = circuit.detector_error_model()


In [ ]:
circuit.diagram('timeline-svg')


In [ ]:
dem.diagram('matchgraph-svg')


In [ ]:
code = CA_5_2_3()
builder = CircuitBuilder(code)
circuit = builder.get_circuit(noise_model='phenomenological', error_rate=ErrorRate(p=0.01), d_rounds=2)
dem = circuit.detector_error_model()


In [ ]:
circuit.diagram('timeline-svg')


In [ ]:
dem.diagram('matchgraph-svg')


In [ ]:
code = CA_5_2_3()
builder = CircuitBuilder(code)
circuit = builder.get_circuit(noise_model='circuit_level', error_rate=ErrorRate(p=0.01, p_meas=0.0, p_idle=0.0), d_rounds=1)
dem = circuit.detector_error_model()


In [ ]:
circuit.diagram('timeline-svg')


In [ ]:
dem.diagram('matchgraph-svg')


In [ ]:
code = CA_9_2_5()
builder = CircuitBuilder(code)
circuit = builder.get_circuit(noise_model='circuit_level', error_rate=ErrorRate(p=0.01, p_meas=0), d_rounds=1)
dem = circuit.detector_error_model()


In [ ]:
circuit.diagram('timeline-svg')


In [ ]:
with open(str(_data_paths.output('circuit_level_round_1_svg')), 'w') as f:
    f.write(str(circuit.diagram('timeline-svg')))


In [ ]:
dem.diagram('matchgraph-svg')


In [ ]:
dem


In [ ]:
dem.num_errors


In [ ]:
code = CA_9_2_5()
builder = CircuitBuilder(code, schedule=1)
circuit = builder.get_circuit(noise_model='circuit_level', error_rate=ErrorRate(p=0.01, p_meas=0), d_rounds=1)
dem = circuit.detector_error_model()


In [ ]:
circuit.diagram('timeline-svg')


In [ ]:
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import visualize_graph_plotly

G, _ = build_tanner_graph_from_dem(dem)
visualize_graph_plotly(G)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from beliefmatching import detector_error_model_to_check_matrices

demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(demMatrices.check_matrix.toarray(), aspect='auto', origin='lower')
ax.set_xticks(np.arange(demMatrices.check_matrix.shape[1]))
ax.set_yticks(np.arange(demMatrices.check_matrix.shape[0]))
ax.set_xlabel('col idx')
ax.set_ylabel('row idx')
fig.colorbar(im, ax=ax)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(code.H, aspect='auto', origin='lower')
ax.set_xticks(np.arange(code.H.shape[1]))
ax.set_yticks(np.arange(code.H.shape[0]))
ax.set_xlabel('col idx')
ax.set_ylabel('row idx')
fig.colorbar(im, ax=ax)


In [ ]:
from utils.graph_struct import build_tanner_graph_from_pcm
from utils.graph_vis import visualize_graph_plotly


In [ ]:
G = build_tanner_graph_from_pcm(code.H, code.stabilizer_coords(), code.qubit_coords())
G.nodes(data=True)


In [ ]:
visualize_graph_plotly(G)
